# 03 — Feature Engineering

**IEEE-CIS Fraud Detection**

Design and test fraud-related features before moving them into `src/features/`.

Feature families:
1. **Time features** from `TransactionDT` (hour, day, weekend)
2. **Amount features** from `TransactionAmt` (log, ratio to card mean)
3. **Identity / device** from `id_31`, `id_33` (browser, resolution)
4. **D-column features** (`D1` raw, grouped deltas)
5. **Aggregation features** (counts and means by `card1`, `ProductCD`)

> Each feature is checked for fraud-rate correlation and leakage risk.

---
## 1 · Imports & Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
FIG_DIR      = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
%matplotlib inline

TARGET = "isFraud"
SECONDS_PER_DAY = 86_400

---
## 2 · Load Cleaned Data

In [ ]:
cleaned_path = INTERIM_DIR / "cleaned_train.parquet"

if cleaned_path.exists():
    df = pd.read_parquet(cleaned_path)
    print(f"Loaded cleaned parquet: {df.shape}")
else:
    print("Cleaned parquet not found — recreating from raw...")
    txn = pd.read_csv(RAW_DIR / "train_transaction.csv")
    idn = pd.read_csv(RAW_DIR / "train_identity.csv")
    df = txn.merge(idn, on="TransactionID", how="left")
    del txn, idn
    # Quick clean: drop >50% missing
    missing_frac = df.isna().mean()
    drop_cols = [c for c in df.columns if missing_frac[c] > 0.50
                 and c not in {"TransactionID", "isFraud", "TransactionDT", "TransactionAmt"}]
    df = df.drop(columns=drop_cols)
    print(f"Recreated cleaned data: {df.shape}")

---
## 3 · Time Features from `TransactionDT`

`TransactionDT` is seconds from an undisclosed reference.  
We extract **hour of day**, **day of week**, **weekend flag**, and **elapsed days**.

**Leakage risk: NONE** — these are derived from the transaction's own timestamp.

In [ ]:
dt = df["TransactionDT"].astype("float64")

df["elapsed_days"]  = dt / SECONDS_PER_DAY
df["hour"]          = ((dt % SECONDS_PER_DAY) // 3600).astype("int8")
df["day_of_week"]   = ((dt // SECONDS_PER_DAY) % 7).astype("int8")
df["is_weekend"]    = df["day_of_week"].isin([5, 6]).astype("int8")
df["is_night"]      = ((df["hour"] >= 0) & (df["hour"] < 6)).astype("int8")

print("Time features created: hour, day_of_week, elapsed_days, is_weekend, is_night")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Fraud rate by hour
fraud_by_hour = df.groupby("hour")[TARGET].mean() * 100
axes[0].bar(fraud_by_hour.index, fraud_by_hour.values, color="#DD4A48", edgecolor="white")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].set_title("Fraud Rate by Hour")

# Fraud rate by day of week
fraud_by_dow = df.groupby("day_of_week")[TARGET].mean() * 100
axes[1].bar(fraud_by_dow.index, fraud_by_dow.values, color="#DD4A48", edgecolor="white")
axes[1].set_xlabel("Day of Week (0=Mon)")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].set_title("Fraud Rate by Day of Week")

# Fraud rate: weekend vs weekday
fraud_by_we = df.groupby("is_weekend")[TARGET].mean() * 100
axes[2].bar(["Weekday", "Weekend"], fraud_by_we.values, color=["#4C72B0", "#DD4A48"], edgecolor="white")
axes[2].set_ylabel("Fraud Rate (%)")
axes[2].set_title("Fraud Rate: Weekday vs Weekend")

fig.suptitle("Time-Feature Signal", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_fe_time_features.png", dpi=150, bbox_inches="tight")
plt.show()

> **Observation:** Fraud rate varies by hour — certain night hours have elevated fraud.  
> Weekend vs weekday also shows a difference. These are cheap, safe features.

---
## 4 · Amount Features from `TransactionAmt`

**Leakage risk: `log1p` and `amt_decimal` are SAFE** (per-row transform).  
**`amt_to_card_mean` requires care** — the group mean must be computed on training data only  
in the final pipeline. Here we explore the idea on the full dataset for illustration.

In [ ]:
amt = df["TransactionAmt"]

# Safe per-row transforms
df["log_TransactionAmt"] = np.log1p(amt.clip(lower=0)).astype("float32")
df["amt_decimal"]        = (amt - amt.astype(int)).astype("float32")
df["amt_is_round"]       = (df["amt_decimal"] == 0).astype("int8")

# Illustration only — in pipeline this must be fit on train
card_mean = df.groupby("card1")["TransactionAmt"].transform("mean")
df["amt_to_card_mean"] = (amt / card_mean.replace(0, np.nan)).astype("float32")

print("Amount features created.")
df[["TransactionAmt", "log_TransactionAmt", "amt_decimal", "amt_is_round", "amt_to_card_mean"]].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Fraud rate for round vs non-round amounts
fr_round = df.groupby("amt_is_round")[TARGET].mean() * 100
axes[0].bar(["Non-Round", "Round"], fr_round.values, color=["#4C72B0", "#DD4A48"], edgecolor="white")
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].set_title("Fraud Rate: Round vs Non-Round Amounts")

# amt_to_card_mean distribution by fraud
for label, color in [(0, "#4C72B0"), (1, "#DD4A48")]:
    subset = df[df[TARGET] == label]["amt_to_card_mean"].dropna().clip(upper=5)
    axes[1].hist(subset, bins=60, alpha=0.6, color=color,
                 label="Legit" if label == 0 else "Fraud", density=True)
axes[1].set_xlabel("TransactionAmt / card1 Mean")
axes[1].set_title("Amount Ratio (amt / card1 mean)")
axes[1].legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "fig_fe_amount_features.png", dpi=150, bbox_inches="tight")
plt.show()

> **Observation:** Fraudulent transactions often involve amounts that deviate from  
> the cardholder's typical spending pattern (`amt_to_card_mean` > 1 is suspicious).  
> Round amounts also show a different fraud rate.

---
## 5 · Identity / Device Features

- `has_identity` — binary flag for identity coverage  
- `browser_family` — extracted from `id_31`  
- `screen_width`, `screen_height` — parsed from `id_33`

**Leakage risk: NONE** — these are row-level transforms.

In [ ]:
# has_identity
id_cols = [c for c in df.columns if c.startswith("id_")] + ["DeviceType", "DeviceInfo"]
id_cols = [c for c in id_cols if c in df.columns]
df["has_identity"] = df[id_cols].notna().any(axis=1).astype("int8")

# Browser family from id_31
if "id_31" in df.columns:
    browser = df["id_31"].fillna("unknown").astype(str).str.lower()
    families = ["chrome", "firefox", "safari", "edge", "opera", "ie", "samsung"]
    df["browser_family"] = "other"
    for fam in families:
        df.loc[browser.str.contains(fam, na=False), "browser_family"] = fam
    df["is_mobile_browser"] = browser.str.contains("mobile|android|iphone|sm-|samsung",
                                                     na=False).astype("int8")

# Screen resolution from id_33
if "id_33" in df.columns:
    parts = df["id_33"].astype(str).str.split("x", expand=True)
    if parts.shape[1] >= 2:
        df["screen_width"]  = pd.to_numeric(parts[0], errors="coerce").astype("float32")
        df["screen_height"] = pd.to_numeric(parts[1], errors="coerce").astype("float32")
        df["screen_area"]   = (df["screen_width"] * df["screen_height"]).astype("float32")

print("Identity features created.")

In [ ]:
if "browser_family" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Fraud rate by browser
    fraud_by_browser = df.groupby("browser_family")[TARGET].agg(["mean", "count"])
    fraud_by_browser = fraud_by_browser[fraud_by_browser["count"] > 100].sort_values("mean", ascending=False)
    axes[0].barh(fraud_by_browser.index, fraud_by_browser["mean"] * 100, color="#DD4A48", edgecolor="white")
    axes[0].set_xlabel("Fraud Rate (%)")
    axes[0].set_title("Fraud Rate by Browser Family")
    
    # Fraud rate: mobile vs desktop browser
    if "is_mobile_browser" in df.columns:
        fr_mob = df.groupby("is_mobile_browser")[TARGET].mean() * 100
        axes[1].bar(["Desktop", "Mobile"], fr_mob.values, color=["#4C72B0", "#E8A838"], edgecolor="white")
        axes[1].set_ylabel("Fraud Rate (%)")
        axes[1].set_title("Fraud Rate: Desktop vs Mobile")
    
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_fe_identity_features.png", dpi=150, bbox_inches="tight")
    plt.show()

> **Observation:** Browser family and mobile vs desktop flags carry signal.  
> Screen resolution further distinguishes device classes.

---
## 6 · D-Column Features

`D1` is commonly interpreted as "days since last activity".  
We also compute `D1_to_elapsed` (ratio of D1 to overall elapsed days).

**Leakage risk: NONE** — these are per-row transforms.

In [ ]:
if "D1" in df.columns:
    df["D1_to_elapsed"] = (df["D1"] / df["elapsed_days"].replace(0, np.nan)).astype("float32")
    
# Count of non-null D columns per row
d_cols = [c for c in df.columns if c.startswith("D") and c[1:].isdigit()]
if d_cols:
    df["D_count_notna"] = df[d_cols].notna().sum(axis=1).astype("int8")
    print(f"D-column features: D1_to_elapsed, D_count_notna (from {len(d_cols)} D-cols)")

In [ ]:
if "D1" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    for i, label in [(0, "Legit"), (1, "Fraud")]:
        sub = df[df[TARGET] == i]["D1"].dropna().clip(upper=200)
        axes[0].hist(sub, bins=60, alpha=0.6, density=True,
                     color="#4C72B0" if i == 0 else "#DD4A48", label=label)
    axes[0].set_xlabel("D1")
    axes[0].set_title("D1 Distribution by Fraud")
    axes[0].legend()
    
    if "D_count_notna" in df.columns:
        fraud_by_d = df.groupby("D_count_notna")[TARGET].mean() * 100
        axes[1].bar(fraud_by_d.index, fraud_by_d.values, color="#DD4A48", edgecolor="white")
        axes[1].set_xlabel("# D-columns non-null")
        axes[1].set_ylabel("Fraud Rate (%)")
        axes[1].set_title("Fraud Rate by D-column Completeness")
    
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_fe_d_features.png", dpi=150, bbox_inches="tight")
    plt.show()

> **Observation:** `D1` has a very different distribution for fraud vs legit transactions.  
> The number of non-null D-columns also correlates with fraud rate — device fingerprint  
> completeness is informative.

---
## 7 · Aggregation Features

Group-level statistics: count, mean amount by `card1` and `ProductCD`.

⚠ **Leakage risk: HIGH** — these aggregations must be **fit on training data only**  
in the final pipeline. Here we compute them on the full dataset for *exploration* purposes  
only. The production implementation in `src/features/aggregation_features.py` uses  
a fit/transform pattern to prevent leakage.

In [ ]:
# card1-level aggregations (illustration — will be refit on train only later)
card1_stats = df.groupby("card1")["TransactionAmt"].agg(["mean", "std", "count"])
card1_stats.columns = ["card1_amt_mean", "card1_amt_std", "card1_count"]
df = df.merge(card1_stats, on="card1", how="left")

# ProductCD-level
if "ProductCD" in df.columns:
    pcd_stats = df.groupby("ProductCD")["TransactionAmt"].agg(["mean", "count"])
    pcd_stats.columns = ["pcd_amt_mean", "pcd_count"]
    df = df.merge(pcd_stats, on="ProductCD", how="left")

print("Aggregation features created (⚠ for exploration only — will refit on train in pipeline).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Fraud rate vs card1 frequency
df["card1_count_bin"] = pd.qcut(df["card1_count"].clip(upper=df["card1_count"].quantile(0.99)),
                                 q=10, duplicates="drop")
fraud_by_freq = df.groupby("card1_count_bin", observed=True)[TARGET].mean() * 100
fraud_by_freq.plot.bar(ax=axes[0], color="#DD4A48", edgecolor="white")
axes[0].set_xlabel("card1 Transaction Count (decile)")
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].set_title("Fraud Rate by card1 Frequency")
axes[0].tick_params(axis="x", rotation=45)

# card1_amt_std — high std may indicate unusual patterns
df["card1_std_bin"] = pd.qcut(df["card1_amt_std"].fillna(0).clip(upper=df["card1_amt_std"].quantile(0.99)),
                               q=10, duplicates="drop")
fraud_by_std = df.groupby("card1_std_bin", observed=True)[TARGET].mean() * 100
fraud_by_std.plot.bar(ax=axes[1], color="#DD4A48", edgecolor="white")
axes[1].set_xlabel("card1 Amount Std (decile)")
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].set_title("Fraud Rate by card1 Amt Std")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
fig.savefig(FIG_DIR / "fig_fe_aggregation_features.png", dpi=150, bbox_inches="tight")
plt.show()

# Clean up temp columns
df.drop(columns=["card1_count_bin", "card1_std_bin"], inplace=True, errors="ignore")

> **Observation:** Cards with very few transactions tend to have higher fraud rates.  
> Cards with high amount variability are also more likely to be associated with fraud.  
> These aggregation features are powerful but must be fit on train only.

---
## 8 · Card Velocity (time-since-last)

Compute time since the previous transaction for each `card1`.  

**Leakage risk: LOW** — this is a backward-looking feature (uses only past data).

In [ ]:
df = df.sort_values("TransactionDT")
df["time_since_last_card1"] = df.groupby("card1")["TransactionDT"].diff().astype("float32")
df["time_since_last_card1"] = df["time_since_last_card1"].fillna(-1)

print(f"Velocity feature created. First-occurrence sentinel = -1.")
print(f"Non-sentinel values: {(df['time_since_last_card1'] > 0).sum():,}")

In [ ]:
# Fraud rate by velocity bin (excluding first-use sentinel)
vel = df[df["time_since_last_card1"] > 0]["time_since_last_card1"] / 3600  # hours
vel_with_target = df[df["time_since_last_card1"] > 0][["time_since_last_card1", TARGET]].copy()
vel_with_target["vel_hours"] = vel_with_target["time_since_last_card1"] / 3600
vel_with_target["vel_bin"] = pd.qcut(vel_with_target["vel_hours"].clip(upper=vel_with_target["vel_hours"].quantile(0.99)),
                                      q=10, duplicates="drop")

fig, ax = plt.subplots(figsize=(10, 4))
fraud_by_vel = vel_with_target.groupby("vel_bin", observed=True)[TARGET].mean() * 100
fraud_by_vel.plot.bar(ax=ax, color="#DD4A48", edgecolor="white")
ax.set_xlabel("Hours Since Last card1 Transaction (decile)")
ax.set_ylabel("Fraud Rate (%)")
ax.set_title("Fraud Rate by Card Velocity")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_fe_card_velocity.png", dpi=150, bbox_inches="tight")
plt.show()

> **Observation:** Very short time gaps between transactions on the same card are  
> strongly associated with fraud — a classic rapid-fire fraud pattern.

---
## 9 · Feature Summary & Leakage Notes

| Feature | Source | Leakage Risk | Notes |
|---------|--------|-------------|-------|
| `hour` | `TransactionDT` | None | Stateless |
| `day_of_week` | `TransactionDT` | None | Stateless |
| `is_weekend` | `TransactionDT` | None | Stateless |
| `is_night` | `TransactionDT` | None | Stateless |
| `elapsed_days` | `TransactionDT` | None | Stateless |
| `log_TransactionAmt` | `TransactionAmt` | None | Stateless |
| `amt_decimal` | `TransactionAmt` | None | Stateless |
| `amt_is_round` | `TransactionAmt` | None | Stateless |
| `amt_to_card_mean` | `TransactionAmt`, `card1` | **HIGH** | Must fit on train only |
| `has_identity` | `id_*` columns | None | Stateless |
| `browser_family` | `id_31` | None | Stateless |
| `is_mobile_browser` | `id_31` | None | Stateless |
| `screen_width/height/area` | `id_33` | None | Stateless |
| `D1_to_elapsed` | `D1`, `TransactionDT` | None | Stateless |
| `D_count_notna` | D-columns | None | Stateless |
| `card1_amt_mean/std/count` | `TransactionAmt`, `card1` | **HIGH** | Must fit on train only |
| `pcd_amt_mean/count` | `TransactionAmt`, `ProductCD` | **HIGH** | Must fit on train only |
| `time_since_last_card1` | `TransactionDT`, `card1` | Low | Backward-looking only |

---
## 10 · Save Feature-Ready Dataset

In [ ]:
output_path = INTERIM_DIR / "feature_ready_train.parquet"

# Drop exploration-only columns and categorical helpers before saving
drop_exploration = ["browser_family"]  # will be re-encoded in pipeline
df_save = df.drop(columns=[c for c in drop_exploration if c in df.columns])

df_save.to_parquet(output_path, index=False, engine="pyarrow")
print(f"Saved feature-ready dataset → {output_path}")
print(f"  Shape: {df_save.shape}")
print(f"  Size : {output_path.stat().st_size / 1e6:.1f} MB")

---
## 11 · Conclusions

### Most promising features
- **Time features** (`hour`, `is_night`) — clear fraud-rate variation with zero leakage risk
- **Amount features** (`log_amt`, `amt_is_round`, `amt_to_card_mean`) — strong signal
- **Card velocity** (`time_since_last_card1`) — very strong fraud indicator
- **D1** — distributional shift between fraud and legit is large
- **has_identity** — simple binary that correlates with fraud

### Leakage warnings
- Aggregation features (`card1_amt_mean`, `pcd_count`, etc.) **must** be fit on training data only
- The `amt_to_card_mean` ratio **must** use train-fitted group means
- All stateless features (time, identity, per-row transforms) are safe pre-split

### Next notebook
→ **`04_train_val_test_split.ipynb`** — chronological split and leakage verification

In [ ]:
print("Notebook 03 complete. ✓")